# Diagnósticos e análises

As consultas distinguem linhas, códigos preenchidos e códigos distintos. A seção de qualidade valida reconciliação Bronze → Prata → Gold, unicidade da chave técnica, integridade referencial e conversões rejeitadas. Execute após as três camadas.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

checks = spark.sql("""
WITH lote_atual AS (
    SELECT source_batch_id
    FROM emendas_parlamentares_bronze
    WHERE source_batch_id IS NOT NULL
    ORDER BY Data_Carga DESC, source_batch_id DESC
    LIMIT 1
),
validacoes AS (
    SELECT 'silver_fact_row_count' AS check_name,
           (SELECT COUNT(*) FROM emendas_parlamentares_prata) AS esperado,
           (SELECT COUNT(*) FROM emendas_parlamentares_fato) AS obtido
    UNION ALL
    SELECT 'silver_only_latest_batch',
           (SELECT COUNT(*) FROM emendas_parlamentares_prata),
           (SELECT COUNT(*) FROM emendas_parlamentares_prata
            WHERE source_batch_id = (SELECT source_batch_id FROM lote_atual))
    UNION ALL
    SELECT 'latest_batch_exists', CAST(1 AS BIGINT),
           (SELECT COUNT(DISTINCT source_batch_id)
            FROM emendas_parlamentares_prata
            WHERE source_batch_id = (SELECT source_batch_id FROM lote_atual))
    UNION ALL
    SELECT 'silver_missing_source_row_id', CAST(0 AS BIGINT),
           (SELECT COUNT(*) FROM emendas_parlamentares_prata WHERE source_row_id IS NULL)
    UNION ALL
    SELECT 'silver_duplicate_source_row_id', CAST(0 AS BIGINT),
           (SELECT COUNT(*) FROM (
                SELECT source_row_id FROM emendas_parlamentares_prata
                GROUP BY source_row_id HAVING COUNT(*) > 1
            ) AS duplicados)
    UNION ALL
    SELECT 'fact_duplicate_source_row_id', CAST(0 AS BIGINT),
           (SELECT COUNT(*) FROM (
                SELECT id_fato_emenda FROM emendas_parlamentares_fato
                GROUP BY id_fato_emenda HAVING COUNT(*) > 1
            ) AS duplicados)
    UNION ALL
    SELECT 'orphan_dimension_keys', CAST(0 AS BIGINT),
           (SELECT COUNT(*) FROM emendas_parlamentares_fato f
            LEFT ANTI JOIN emendas_parlamentares_dim_tempo d ON f.sk_tempo = d.sk_tempo)
         + (SELECT COUNT(*) FROM emendas_parlamentares_fato f
            LEFT ANTI JOIN emendas_parlamentares_dim_autor d ON f.sk_autor = d.sk_autor)
         + (SELECT COUNT(*) FROM emendas_parlamentares_fato f
            LEFT ANTI JOIN emendas_parlamentares_dim_localidade d ON f.sk_localidade = d.sk_localidade)
         + (SELECT COUNT(*) FROM emendas_parlamentares_fato f
            LEFT ANTI JOIN emendas_parlamentares_dim_programa_orcamentario d
                ON f.sk_programa_orcamentario = d.sk_programa_orcamentario)
    UNION ALL
    SELECT 'invalid_conversions', CAST(0 AS BIGINT),
           (SELECT COUNT(*) FROM emendas_parlamentares_rejeicoes)
    UNION ALL
    SELECT 'unknown_dimension_members', CAST(4 AS BIGINT),
           (SELECT COUNT(*) FROM emendas_parlamentares_dim_tempo WHERE sk_tempo = 'UNKNOWN')
         + (SELECT COUNT(*) FROM emendas_parlamentares_dim_autor WHERE sk_autor = 'UNKNOWN')
         + (SELECT COUNT(*) FROM emendas_parlamentares_dim_localidade WHERE sk_localidade = 'UNKNOWN')
         + (SELECT COUNT(*) FROM emendas_parlamentares_dim_programa_orcamentario
            WHERE sk_programa_orcamentario = 'UNKNOWN')
)
SELECT check_name, esperado, obtido,
       CASE WHEN esperado = obtido THEN 'OK' ELSE 'FALHA' END AS status
FROM validacoes
""").collect()

for check in checks:
    print(f"{check['status']}: {check['check_name']} (esperado={check['esperado']}, obtido={check['obtido']})")
falhas = [check["check_name"] for check in checks if check["esperado"] != check["obtido"]]
if falhas:
    raise ValueError(f"Falhas nas validações de qualidade: {', '.join(falhas)}")

In [ ]:
# Métricas do snapshot: linhas não são sinônimo de códigos de emenda distintos.
spark.sql("""
SELECT
    COUNT(*) AS total_linhas,
    COUNT(codigo_emenda) AS linhas_com_codigo_emenda,
    COUNT(DISTINCT codigo_emenda) AS codigos_emenda_distintos,
    SUM(CASE WHEN codigo_emenda IS NULL THEN 1 ELSE 0 END) AS linhas_sem_codigo
FROM emendas_parlamentares_prata
""").show(truncate=False)

spark.sql("""
SELECT source_batch_id, total_registros_origem, total_conversoes_invalidas,
       total_sentinelas_normalizadas, data_verificacao
FROM emendas_parlamentares_quality_metrics
""").show(truncate=False)

In [ ]:
# Códigos nulos são contabilizados separadamente; não entram no diagnóstico de duplicidade.
spark.sql("""
SELECT COUNT(*) AS linhas_sem_codigo_emenda
FROM emendas_parlamentares_prata
WHERE codigo_emenda IS NULL
""").show()

spark.sql("""
SELECT codigo_emenda, COUNT(*) AS ocorrencias
FROM emendas_parlamentares_prata
WHERE codigo_emenda IS NOT NULL
GROUP BY codigo_emenda
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC, codigo_emenda
""").show(20, truncate=False)

In [ ]:
# Contagem de linhas e de códigos válidos por autor.
spark.sql("""
SELECT
    codigo_autor,
    nome_autor,
    COUNT(*) AS total_linhas,
    COUNT(codigo_emenda) AS linhas_com_codigo,
    COUNT(DISTINCT codigo_emenda) AS codigos_distintos
FROM emendas_parlamentares_prata
GROUP BY codigo_autor, nome_autor
ORDER BY total_linhas DESC
LIMIT 10
""").show(truncate=False)

# Localidades desconhecidas permanecem visíveis como UNKNOWN na dimensão.
spark.sql("""
SELECT
    l.uf AS estado,
    COUNT(*) AS total_linhas,
    COUNT(f.codigo_emenda) AS linhas_com_codigo,
    SUM(f.valor_empenhado) AS total_empenhado,
    SUM(f.valor_pago) AS total_pago
FROM emendas_parlamentares_fato f
JOIN emendas_parlamentares_dim_localidade l
    ON f.sk_localidade = l.sk_localidade
GROUP BY l.uf
ORDER BY total_linhas DESC
""").show(20, truncate=False)

In [ ]:
spark.sql("""
SELECT ano_emenda,
       COUNT(*) AS total_linhas,
       COUNT(codigo_emenda) AS linhas_com_codigo,
       DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
FROM emendas_parlamentares_prata
GROUP BY ano_emenda
ORDER BY total_linhas DESC
""").show(truncate=False)

spark.sql("""
SELECT nome_funcao,
       COUNT(*) AS total_linhas,
       SUM(valor_empenhado) AS total_empenhado,
       SUM(valor_pago) AS total_pago,
       DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
FROM emendas_parlamentares_prata
GROUP BY nome_funcao
ORDER BY total_linhas DESC
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT campo, valor_original, COUNT(*) AS ocorrencias
FROM emendas_parlamentares_rejeicoes
GROUP BY campo, valor_original
ORDER BY ocorrencias DESC, campo
""").show(50, truncate=False)